# Avocado Memo: Arithmetic Verification

Recomputes every simple-arithmetic figure in *Memo: U.S. Avocado Market Trends, 2025 – 2026 YTD* from the Hass Avocado Board weekly files, then saves a copy of the memo with a green check mark next to each figure that matches (and a yellow highlight on any that doesn't).

Requires `pip install pandas pymupdf`. Change `UP` to the folder holding the CSVs and memo PDF.

## 1. Load the data
Reads only the 9 columns used. Dollars are rebuilt as average price × units, since the files have no dollar column.

In [1]:
import pandas as pd, pymupdf

UP, U = '/mnt/user-data/uploads/', 'Total Bulk and Bags'
C = [U, 'dol', '4046', '4225', '4770', 'TotalBagged']
d = pd.concat(pd.read_csv(f'{UP}HAB-FlatFile-Weekly-{y}.csv',
                          usecols=['Geography', 'Type', 'Period', 'ASP Current Year', *C[:1], *C[2:]])
              .assign(yr=int(y[:4])) for y in ('2025', '2026-P1-P8'))
d['dol'] = d['ASP Current Year'] * d[U]

## 2. Aggregate once
One `groupby` builds a small table (region × type × year × week). Every figure below is sliced from it. `S()` sums any region, week range, and type; `px()` is the unit-weighted average price; `pct()`/`yoy()` are percent changes.

In [2]:
# One groupby pass; every figure below is sliced from this small, sorted table
g  = d.groupby(['Geography', 'Type', 'yr', 'Period'])[C].sum()
gt = g.groupby(level=[0, 2, 3]).sum()                        # conventional + organic
IX = pd.IndexSlice
S  = lambda geo, a=1, b=32, t=None: (g.loc[IX[geo, t, :, a:b], :] if t else
                                     gt.loc[IX[geo, :, a:b], :]).groupby(level='yr').sum()
px  = lambda t: t['dol'] / t[U]
pct = lambda a, b: 100 * (a / b - 1)
yoy = lambda t, c=U: pct(t.loc[2026, c], t.loc[2025, c])

TU = 'Total U.S.'
T, E, L8, CV, OR = S(TU), S(TU, b=19), S(TU, 25), S(TU, t='Conventional'), S(TU, t='Organic')
P = px(T)
wk = gt.loc[TU]
wu, wp = wk[U].unstack(0), px(wk).unstack(0)
l8 = lambda r: yoy(S(r, 25))

## 3. Figures to check
Each row: page, the text as it appears in the memo, the memo's number, the recomputed number, and the allowed rounding margin (half of the memo's last shown digit; 1 point for "roughly 25%", 1.9–2.1× for "doubled").

In [3]:
# (page, text on page, figure in memo, recomputed, tolerance)
checks = [
 (0, 'Units up 13.6%', 13.6, yoy(T), .05),
 (0, '2.13 billion', 2.13, T.loc[2026, U] / 1e9, .005),
 (0, '1.87 billion', 1.87, T.loc[2025, U] / 1e9, .005),
 (0, 'Price down 19%', -19, pct(P[2026], P[2025]), .5),
 (0, '$1.01', 1.01, P[2026], .005),
 (0, '$1.25', 1.25, P[2025], .005),
 (0, 'Dollars down 8%', -8, yoy(T, 'dol'), .5),
 (0, '$2.14 billion', 2.14, T.loc[2026, 'dol'] / 1e9, .005),
 (0, '$2.33 billion', 2.33, T.loc[2025, 'dol'] / 1e9, .005),
 (0, 'slowed to 6%', 6, yoy(L8), .5),
 (1, '(up 11%)', 11, pct(wu.loc[6, 2026], wu.loc[6, 2025]), .5),
 (1, '(up 10%)', 10, pct(wu.loc[18, 2026], wu.loc[18, 2025]), .5),
 (2, 'roughly 25%', -25, pct(px(E)[2026], px(E)[2025]), 1),
 (2, 'rose 23%', 23, pct(wp.loc[26, 2026], wp.loc[20, 2026]), .5),
 (2, 'still 8% below', -8, pct(wp.loc[32, 2026], wp.loc[32, 2025]), .5),
 (2, '(+1% units)', 1, l8('California'), .5),
 (2, '(+1% units)', 1, l8('West'), .5),
 (2, 'stayed up 15%', 15, l8('Southeast'), .5),
 (2, 'units +14.5%', 14.5, yoy(CV), .05),
 (2, '(-0.7%)', -.7, yoy(OR), .05),
 (2, '10% price cut', -10, pct(px(OR)[2026], px(OR)[2025]), .5),
 (2, '5.7%', 5.7, 100 * OR.loc[2025, U] / T.loc[2025, U], .05),
 (2, '4.9%', 4.9, 100 * OR.loc[2026, U] / T.loc[2026, U], .05),
 (2, 'rose 17.6%', 17.6, yoy(T, 'TotalBagged'), .05),
 (2, '41.8%', 41.8, 100 * T.loc[2025, 'TotalBagged'] / T.loc[2025, U], .05),
 (2, '43.3%', 43.3, 100 * T.loc[2026, 'TotalBagged'] / T.loc[2026, U], .05),
 (2, 'rose 36%', 36, yoy(T, '4225'), .5),
 (2, 'doubled', 2, T.loc[2026, '4770'] / T.loc[2025, '4770'], .1),
 (2, 'rose only 2.5%', 2.5, yoy(T, '4046'), .05),
]

## 4. Open the memo
Each page's text is extracted once and reused for every search; check marks for a page are drawn on one layer.

In [4]:
doc = pymupdf.open(UP + 'Memo_U_S__Avocado_Market_Trends__2025___2026_YTD.pdf')
pg = list(doc)
tp = [p.get_textpage() for p in pg]                          # extract each page's text once
sh = [p.new_shape() for p in pg]                             # one drawing layer per page
find = lambda p, t: pg[p].search_for(t, textpage=tp[p])
tick = lambda p, r: sh[p].draw_polyline([(r.x1 + .5, r.y0 + .5), (r.x1 + 2.3, r.y0 + 2.5), (r.x1 + 5.5, r.y0 - 2.5)])
hl = lambda p, rects: [pg[p].add_highlight_annot(r).set_colors(stroke=(1, .9, 0)) for r in rects]

## 5. Compare the text figures

In [5]:
rows, done = [], set()
for p, txt, claim, calc, tol in checks:
    ok = abs(claim - calc) <= tol
    rows.append((p + 1, txt, claim, round(calc, 4), 'OK' if ok else 'MISMATCH'))
    if (p, txt) in done: continue
    done.add((p, txt))
    tick(p, find(p, txt)[0]) if ok else hl(p, find(p, txt))

## 6. Compare the regional table
Reads each row's four numbers straight off page 3 and compares them to the recomputed values.

In [6]:
# Region table (page 3): read each row's numbers off the PDF and compare every cell
words = pg[2].get_text('words', textpage=tp[2])
x_col = next(w[0] for w in words if w[4] == 'Region')
for r in ['Southeast', 'California', 'Midsouth', 'South Central', 'Northeast', 'Great Lakes', 'Plains', 'West']:
    lab = next(w for w in words if w[4] == r.split()[0] and abs(w[0] - x_col) < 3)
    cells = [w for w in words if abs(w[1] - lab[1]) < 3 and w[0] > lab[0] + 60][-4:]
    t = S(r); pr = px(t)
    for c, v, tol in zip(cells, (t.loc[2026, U] / 1e6, yoy(t), pr[2026], pct(pr[2026], pr[2025])),
                         (.05, .05, .005, .05)):
        claim, rect = float(c[4].strip('+$%')), pymupdf.Rect(c[:4])
        ok = abs(claim - v) <= tol
        rows.append((3, f'{r}: {c[4]}', claim, round(v, 4), 'OK' if ok else 'MISMATCH'))
        tick(2, rect) if ok else hl(2, [rect])

## 7. Save and show results

In [7]:
for s_ in sh: s_.finish(color=(0, .6, .2), width=1.3, closePath=False); s_.commit()
doc.save('/mnt/user-data/outputs/Memo_Avocado_Trends_VERIFIED.pdf', garbage=3, deflate=True)
print(pd.DataFrame(rows, columns=['page', 'figure', 'memo', 'python', 'result']).to_string(index=False))

 page                figure   memo   python result
    1        Units up 13.6%  13.60  13.6367     OK
    1          2.13 billion   2.13   2.1258     OK
    1          1.87 billion   1.87   1.8707     OK
    1        Price down 19% -19.00 -19.0516     OK
    1                 $1.01   1.01   1.0079     OK
    1                 $1.25   1.25   1.2452     OK
    1       Dollars down 8%  -8.00  -8.0129     OK
    1         $2.14 billion   2.14   2.1427     OK
    1         $2.33 billion   2.33   2.3293     OK
    1          slowed to 6%   6.00   5.7193     OK
    2              (up 11%)  11.00  10.8627     OK
    2              (up 10%)  10.00  10.1774     OK
    3           roughly 25% -25.00 -24.0690     OK
    3              rose 23%  23.00  23.1525     OK
    3        still 8% below  -8.00  -7.5159     OK
    3           (+1% units)   1.00   1.2052     OK
    3           (+1% units)   1.00   1.1579     OK
    3         stayed up 15%  15.00  15.4272     OK
    3          units +14.5%  14